In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
data_path = Path("final_dataset.parquet")
data = pd.read_parquet(data_path)

## Data Split
 

In [3]:
for col in ["news_date_ny", "entry_date", "exit_date"]:
    data[col] = pd.to_datetime(data[col])

in_period = (data["news_date_ny"].ge("2020-01-01") & data["news_date_ny"].lt("2025-01-01"))
has_target = data["excess_return_5d"].notna()
eligible = data.loc[in_period & has_target].copy()
train_mask = eligible["news_date_ny"].lt("2023-01-01")
val_mask = (eligible["news_date_ny"].ge("2023-01-01") & eligible["news_date_ny"].lt("2024-01-01"))
test_mask = eligible["news_date_ny"].ge("2024-01-01")
train = eligible.loc[train_mask & eligible["exit_date"].lt("2023-01-01")].copy()
validation = eligible.loc[val_mask & eligible["exit_date"].lt("2024-01-01")].copy()
test = eligible.loc[test_mask].copy()
split_dir = data_path.parent / "splits"
split_dir.mkdir(parents=True, exist_ok=True)
summary = []
for name, frame in [("train", train),("validation", validation),("test", test),]:
    frame.to_parquet(split_dir / f"{name}_unprocessed.parquet",index=False,)
    summary.append({
        "split": name,
        "rows": len(frame),
        "stocks": frame["PERMNO"].nunique(),
        "first_news_date": frame["news_date_ny"].min(),
        "last_news_date": frame["news_date_ny"].max(),
        "last_exit_date": frame["exit_date"].max(),})
print("News outside 2020-2024:", (~in_period).sum())
print("Missing targets within period:", (in_period & ~has_target).sum())
print("Training rows purged:", train_mask.sum() - len(train))
print("Validation rows purged:", val_mask.sum() - len(validation))
display(pd.DataFrame(summary))

News outside 2020-2024: 5
Missing targets within period: 441
Training rows purged: 308
Validation rows purged: 751


,split,rows,stocks,first_news_date,last_news_date,last_exit_date
0,train,258946,388,2020-01-01,2022-12-22,2022-12-30
1,validation,89502,390,2023-01-02,2023-12-21,2023-12-29
2,test,100380,397,2024-01-01,2024-12-23,2024-12-31


## Imputation and Scaling

In [4]:
missing_pct = train.isna().mean().mul(100).sort_values(ascending=False)
display(missing_pct[missing_pct > 0].rename("missing_percent").to_frame())

,missing_percent
DlyRetMissFlg,99.997297
leverage,4.919559
operating_profitability,0.937647
distance_from_annual_high,0.164127
beta_spy_252d,0.163355
idiosyncratic_volatility_252d,0.163355
sales_growth,0.138639
momentum_12_2,0.134391
asset_growth,0.093070
return_on_assets,0.093070


In [ ]:
# Remove the column consistently from all three splits
train = train.drop(columns=["DlyRetMissFlg"])
validation = validation.drop(columns=["DlyRetMissFlg"])
test = test.drop(columns=["DlyRetMissFlg"])


['rpa_date_utc', 'timestamp_utc', 'rp_story_id', 'rp_entity_id', 'entity_type', 'entity_name', 'relevance', 'event_sentiment_score', 'event_relevance', 'event_similarity_key', 'event_similarity_days', 'fact_level', 'category', 'rp_story_event_index', 'event_predictor', 'opinion_label_flag', 'is_fact', 'is_forecast', 'is_opinion', 'novel_1day', 'novel_7days', 'earnings_beat', 'earnings_miss', 'revenue_beat', 'revenue_miss', 'earnings_guidance', 'dividend_changes', 'share_repurchases', 'equity_issuance', 'debt_issuance', 'credit_rating_changes', 'default_bankruptcy', 'mergers_acquisitions', 'asset_purchases_sales', 'analyst_recommendation_changes', 'analyst_price_target_changes', 'management_changes', 'insider_transactions', 'lawsuits', 'regulatory_actions', 'product_launches', 'product_recalls', 'partnerships_contracts', 'earnings_beat_sentiment', 'earnings_miss_sentiment', 'revenue_beat_sentiment', 'revenue_miss_sentiment', 'earnings_guidance_sentiment', 'dividend_changes_sentiment', '

In [9]:
stock_features = [
    "log_market_cap", "book_to_market", "earnings_to_price",
    "sales_to_price", "dividend_yield_12m", "leverage",
    "return_on_assets", "operating_profitability",
    "asset_growth", "sales_growth",
    "return_1d", "return_5d", "return_21d", "momentum_12_2",
    "volatility_20d", "volatility_60d",
    "beta_spy_252d", "idiosyncratic_volatility_252d",
    "share_turnover_20d", "abnormal_volume",
    "log_dollar_volume_20d", "illiquidity_20d",
    "distance_from_annual_high",
    "market_return_5d", "market_volatility_20d", "relative_spread"
]

event_features = [
    "earnings_beat", "earnings_miss",
    "revenue_beat", "revenue_miss", "earnings_guidance",
    "dividend_changes", "share_repurchases", "equity_issuance",
    "debt_issuance", "credit_rating_changes", "default_bankruptcy",
    "mergers_acquisitions", "asset_purchases_sales",
    "analyst_recommendation_changes", "analyst_price_target_changes",
    "management_changes", "insider_transactions", "lawsuits",
    "regulatory_actions", "product_launches", "product_recalls",
    "partnerships_contracts"
]
news_features = [
    "relevance", "event_sentiment_score", "event_relevance",
    "event_similarity_days", "is_fact", "is_forecast", "is_opinion","novel_1day", "novel_7days"
] + event_features + [f"{event}_sentiment" for event in event_features]
all_features = stock_features + news_features
target = "excess_return_5d"

### Imputation by median

Several stock features can be skewed or contain extreme values, such as leverage, trading volume, and valuation ratios. The median is less affected by those extremes than the mean.

In [10]:
X_train = train[all_features].copy()
X_val = validation[all_features].copy()
X_test = test[all_features].copy()
y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)
print("Training missing values:", X_train.isna().sum().sum())
print("Validation missing values:", X_val.isna().sum().sum())
print("Test missing values:", X_test.isna().sum().sum())

Training missing values: 0
Validation missing values: 0
Test missing values: 0


In [11]:
from sklearn.preprocessing import StandardScaler
# scaled by mean and SD
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [12]:

for name, X, y in [
    ("Train", X_train_scaled, y_train),("Validation", X_val_scaled, y_val),("Test", X_test_scaled, y_test),]:
    print(
        name,"X shape:", X.shape,"y shape:", y.shape,"All finite:", np.isfinite(X).all() and np.isfinite(y).all())

Train X shape: (258946, 79) y shape: (258946,) All finite: True
Validation X shape: (89502, 79) y shape: (89502,) All finite: True
Test X shape: (100380, 79) y shape: (100380,) All finite: True


## Baseline Model

In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict the training target mean for every validation row
baseline_pred = np.full(len(y_val), y_train.mean())
baseline_results = pd.Series({
    "Validation RMSE": np.sqrt(mean_squared_error(y_val, baseline_pred)),
    "Validation MAE": mean_absolute_error(y_val, baseline_pred)})
display(baseline_results)

Validation RMSE    0.034158
Validation MAE     0.024462
dtype: float64